# Dataset Exploration

In [17]:
from pathlib import Path 
Path.cwd()

WindowsPath('c:/Users/kwsta/OneDrive/Desktop/Desetation project/brain-tumour-mri-classification/notebooks')

In [18]:
from pathlib import Path

# creating a path to the data folder
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Data folder exists:", DATA_DIR.exists())

Project root: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification
Data folder: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\data
Data folder exists: True


In [19]:
print("Data folder contents:", list(DATA_DIR.iterdir()))

for file in DATA_DIR.iterdir():
    if file.is_file():
        print(f"File: {file.name}, Size: {file.stat().st_size} bytes")
    elif file.is_dir():
        print(f"Directory: {file.name}")


Data folder contents: [WindowsPath('c:/Users/kwsta/OneDrive/Desktop/Desetation project/brain-tumour-mri-classification/data/Testing'), WindowsPath('c:/Users/kwsta/OneDrive/Desktop/Desetation project/brain-tumour-mri-classification/data/Training')]
Directory: Testing
Directory: Training


In [20]:
TRAINING_DIR = DATA_DIR / "Training"

# Check what the training directory contains
for folder in sorted(TRAINING_DIR.iterdir()):
    if folder.is_dir():
        print(folder.name)

glioma
meningioma
notumor
pituitary


In [21]:
# Count the number of images in each class folder
for class_folder in sorted(TRAINING_DIR.iterdir()):
    if class_folder.is_dir():
        image_count = sum(1 for item in class_folder.iterdir() if item.is_file())
        print(f"{class_folder.name}: {image_count} images")

glioma: 1400 images
meningioma: 1400 images
notumor: 1400 images
pituitary: 1400 images


In [22]:
TESTING_DIR = DATA_DIR / "Testing"

# Check what the testing directory contains
for class_folder in sorted(TESTING_DIR.iterdir()):
    if class_folder.is_dir():
        image_count = sum(
            1 for item in class_folder.iterdir()
            if item.is_file()
        )
        print(f"{class_folder.name}: {image_count} images")

glioma: 400 images
meningioma: 400 images
notumor: 400 images
pituitary: 400 images


In [23]:
from collections import Counter

# Count the number of images in each class folder and their file extensions
file_extensions = Counter()

for split_folder in [TRAINING_DIR, TESTING_DIR]:
    for class_folder in split_folder.iterdir():
        if class_folder.is_dir():
            for image_file in class_folder.iterdir():
                if image_file.is_file():
                    file_extensions[image_file.suffix.lower()] += 1

print(file_extensions)

Counter({'.jpg': 7200})


In [24]:
from collections import Counter
from dbm import error
from PIL import Image

image_sizes = Counter()
unreadable_images = []

# Count the number of images in each class folder and their sizes
for split_folder in [TRAINING_DIR, TESTING_DIR]:
    for class_folder in split_folder.iterdir():
        if class_folder.is_dir():
            for image_file in class_folder.iterdir():
                if image_file.is_file():
                    try:
                        with Image.open(image_file) as img:
                            image_sizes[img.size] += 1
                    except Exception as e:
                        print(f"Error opening {image_file}: {e}")
                        unreadable_images.append((image_file, e))
                        
print("Image sizes:", image_sizes)

Image sizes: Counter({(512, 512): 5014, (225, 225): 338, (630, 630): 90, (201, 251): 57, (228, 221): 51, (232, 217): 50, (442, 442): 48, (236, 236): 48, (150, 198): 44, (200, 252): 43, (428, 417): 42, (227, 222): 39, (173, 201): 36, (206, 244): 35, (256, 256): 33, (192, 192): 31, (201, 250): 29, (218, 231): 29, (215, 234): 28, (227, 262): 27, (468, 444): 24, (208, 242): 24, (504, 540): 23, (359, 449): 23, (214, 236): 22, (300, 168): 21, (400, 442): 20, (236, 213): 19, (393, 400): 18, (207, 243): 18, (442, 454): 17, (230, 282): 17, (276, 326): 17, (441, 442): 16, (235, 214): 16, (550, 664): 16, (194, 259): 15, (680, 680): 15, (420, 280): 14, (339, 340): 13, (212, 238): 12, (275, 183): 12, (642, 361): 12, (196, 257): 12, (380, 530): 12, (350, 350): 11, (177, 197): 9, (356, 474): 9, (350, 393): 7, (554, 554): 6, (455, 500): 5, (236, 357): 5, (1024, 1024): 5, (235, 227): 5, (208, 248): 5, (220, 275): 5, (210, 240): 4, (200, 223): 4, (341, 395): 4, (212, 237): 4, (275, 301): 4, (416, 512): 

In [ ]:
# Checking for unreadable images
print(f"unreadable_images: {len(unreadable_images)}")

unreadable_images: 0


In [25]:
# Checking how the images are stored in the folders

image_modes = Counter()
for split_folder in [TRAINING_DIR, TESTING_DIR]:
    for class_folder in split_folder.iterdir():
        if class_folder.is_dir():
            for image_file in class_folder.iterdir():
                if image_file.is_file():
                    try:
                        with Image.open(image_file) as img:
                            image_modes[img.mode] += 1
                    except Exception as e:
                        print(f"Error opening {image_file}: {e}")
                        unreadable_images.append((image_file, e))

print("Image modes:", image_modes)

Image modes: Counter({'RGB': 4129, 'L': 3067, 'RGBA': 3, 'P': 1})


In [ ]:
# Checking if the MRI images are grayscale images saved as RGB format or if they are actually RGB

import numpy as np

grayscale_rgb_images = [] # List to store RGB images
for split_folder in [TRAINING_DIR, TESTING_DIR]:
    for class_folder in split_folder.iterdir():
        if class_folder.is_dir():
            for image_file in class_folder.glob("*.jpg"):  # Assuming images are in .jpg format; adjust as needed
                with Image.open(image_file) as img:
                    if img.mode == 'RGB':
                        
                        # Convert the image to a NumPy array and check if all channels are equal (indicating grayscale)
                        pixels = np.array(img, dtype=np.int16)
                        
                        # For every pixel, check if R == G == B
                        if np.all(pixels[..., 0] == pixels[..., 1]) and np.all(pixels[..., 1] == pixels[..., 2]): # The ... indicates that we are checking all pixels in the image
                            grayscale_rgb_images.append(image_file)
                        
print(f"Number of grayscale images saved as RGB: {len(grayscale_rgb_images)}")
                            


Number of grayscale images saved as RGB: 4003
